# Publish Leonida-PC to GitHub from Colab (UNTESTED against real GitHub)
**Token safety:** never paste your token into chat, a cell, or a file. Use Colab Secrets (key icon in the left bar): add a secret named `GH_TOKEN`, enable "Notebook access". Best practice: a **fine-grained token limited to the single repo** `Leonida-PC` with *Contents: read & write* (create the empty public repo `Leonida-PC` in the GitHub web UI first, 10 seconds). Creating repos via token needs a classic token with `repo` scope: use it only if you must, and revoke it afterwards.

In [ ]:
from google.colab import files
files.upload()   # choose Leonida-PC.zip
import zipfile, os
z = zipfile.ZipFile('Leonida-PC.zip')
for i in z.infolist():
    z.extract(i, '.')
    m = (i.external_attr >> 16) & 0o777
    if m and not i.is_dir(): os.chmod(i.filename, m)
%cd /content/Leonida-PC

In [ ]:
from google.colab import userdata
import os
try:
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
except Exception:
    import getpass
    os.environ['GH_TOKEN'] = getpass.getpass('GitHub token (hidden, not stored): ')
import subprocess
def pub(*args):
    return subprocess.run(['python3', 'tools/publish/publish.py', *args], env=os.environ).returncode
print('rc', pub('--whoami'))   # prints your login and token scopes only

Dry run: checks `make check`, whether the repo exists, and what would be rewritten. Changes nothing.

In [ ]:
print('rc', pub('--dry-run', '--rewrite-identity'))   # add --owner <org> to publish under an organisation

**Real publish.** `--rewrite-identity` replaces the placeholder author with your GitHub no-reply identity (no real email is exposed) and re-creates tag v0.1.0-rc1. Review the dry-run output first.

In [ ]:
print('rc', pub('--yes', '--rewrite-identity'))

In [ ]:
del os.environ['GH_TOKEN']   # then revoke/delete the token at github.com/settings/tokens if it was temporary